# Beca 18 RAG Chatbot
**HW03 · Applied Data Science**

RAG chatbot that answers questions about Beca 18 regulations (PRONABEC RDE 033-2026)  
using Gemini embeddings, ChromaDB for vector storage, and grounded generation.

---
## Section 1 — Install Dependencies

In [ ]:
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules

# In Colab: clone the repo (which includes the PDF in data/) and set it as
# the working directory for all subsequent cells.
# Locally: requirements.txt is already present — skip the clone.
if IN_COLAB:
    repo_dir = "beca18-rag-chatbot"
    if not os.path.isdir(repo_dir):
        subprocess.run(
            ["git", "clone", "https://github.com/Erics-20/beca18-rag-chatbot.git"],
            check=True,
        )
    os.chdir(repo_dir)
    print(f"Working directory set to: {os.getcwd()}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"],
    check=True,
)
print("Installation complete. If running in Colab, restart the session now (Runtime > Restart session).")

---
## Section 2 — Load API Key from `.env`

**In Google Colab:** run the cell below — it will prompt you to upload your `.env` file.  
**Locally (Jupyter):** place `.env` in the same folder as this notebook and run the cell.

The `.env` file must contain:
```
GEMINI_API_KEY=your_actual_key_here
```

> The key is never stored in the notebook. The `.env` file is listed in `.gitignore`.

In [ ]:
import sys
import os
from dotenv import load_dotenv

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import files
    print("Running in Colab — please upload your .env file when prompted.")
    uploaded = files.upload()
    if ".env" not in uploaded:
        raise FileNotFoundError("No .env file was uploaded. Please upload a file named exactly '.env'.")
    with open(".env", "wb") as f:
        f.write(uploaded[".env"])
    print(".env file saved to the Colab working directory.")
else:
    print("Running locally — expecting .env in the notebook directory.")

load_dotenv(dotenv_path=".env", override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise EnvironmentError(
        "GEMINI_API_KEY not found. "
        "Make sure your .env file contains: GEMINI_API_KEY=your_key_here"
    )

print(f"GEMINI_API_KEY loaded successfully (starts with: {GEMINI_API_KEY[:6]}...).")

---
## Section 3 — Confirm Environment

Print the installed package versions to verify the environment is correctly set up.

In [ ]:
import importlib.metadata
import platform

packages = [
    "pypdf",
    "tiktoken",
    "langchain-text-splitters",
    "google-genai",
    "chromadb",
    "ipywidgets",
    "tqdm",
    "python-dotenv",
]

print(f"Python            : {platform.python_version()}")
print("-" * 40)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        status = "OK"
    except importlib.metadata.PackageNotFoundError:
        version = "NOT INSTALLED"
        status = "MISSING"
    print(f"{pkg:<30} {version:<12} [{status}]")

print("-" * 40)
print("Environment check complete.")

---
## Section 4 — Extract and Clean PDF Text

`beca18_reglamento.pdf` is committed to the repository inside `data/`, so it is available
automatically after the `git clone` in Section 1 — no upload or Drive access needed.

**Cleaning steps applied per page:**
1. Repeated lines detected across pages (headers/footers) are removed.
2. Isolated line breaks (single `\n` inside a paragraph) are collapsed into spaces.
3. Multiple consecutive spaces/tabs are collapsed into one.
4. A `[PAGE N]` marker is prepended to each page.

In [ ]:
import re
from pathlib import Path
from pypdf import PdfReader

# ── Configuration ──────────────────────────────────────────────────────────────
HEADER_FOOTER_LINES = 2    # lines to inspect at top/bottom of each page
REPEAT_THRESHOLD    = 0.4  # a line appearing on ≥40 % of pages → header/footer
PDF_NAME            = "beca18_reglamento.pdf"

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

PDF_PATH = data_dir / PDF_NAME
if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"{PDF_NAME} not found in data/. "
        "Make sure it is committed to the repository and the clone is up to date."
    )
print(f"PDF path: {PDF_PATH}")


# ── Helper: detect repeated header/footer lines ────────────────────────────────
def detect_repeated_lines(pages_raw: list, n_edge: int, threshold: float) -> set:
    from collections import Counter
    candidates = Counter()
    for text in pages_raw:
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        edge = lines[:n_edge] + lines[-n_edge:]
        for line in set(edge):
            candidates[line] += 1
    total = len(pages_raw)
    return {line for line, count in candidates.items() if count / total >= threshold}


# ── Helper: clean one page ─────────────────────────────────────────────────────
def clean_page(text: str, repeated: set) -> str:
    lines = [l for l in text.splitlines() if l.strip() not in repeated]
    text = "\n".join(lines)
    # Collapse isolated line breaks (single \n → space, keep paragraph breaks \n\n)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    # Collapse multiple spaces/tabs into one
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


# ── Extract raw text page by page ─────────────────────────────────────────────
reader    = PdfReader(PDF_PATH)
pages_raw = [page.extract_text() or "" for page in reader.pages]
print(f"Pages extracted: {len(pages_raw)}")

# ── Detect repeated lines (headers / footers) ─────────────────────────────────
repeated = detect_repeated_lines(pages_raw, HEADER_FOOTER_LINES, REPEAT_THRESHOLD)
if repeated:
    print(f"\nDetected {len(repeated)} repeated header/footer line(s) — will be removed:")
    for line in sorted(repeated):
        print(f"  \u2022 {line!r}")
else:
    print("\nNo repeated headers/footers detected.")

# ── Clean pages and prepend [PAGE N] markers ──────────────────────────────────
pages_clean = []
for i, raw in enumerate(pages_raw, start=1):
    cleaned = clean_page(raw, repeated)
    pages_clean.append(f"[PAGE {i}]\n{cleaned}")

full_text = "\n\n".join(pages_clean)

# ── Stats ──────────────────────────────────────────────────────────────────────
char_count = len(full_text)
word_count = len(full_text.split())

print(f"\n{'\u2500'*40}")
print(f"Pages  : {len(pages_clean):>10,}")
print(f"Chars  : {char_count:>10,}")
print(f"Words  : {word_count:>10,}")
print(f"{'\u2500'*40}")

---
## Section 5 — Tokenisation & Chunking

### Why 400 tokens with 60-token overlap?

The Gemini embedding model (`gemini-embedding-001`) accepts up to **8,192 tokens** per input.  
Choosing `chunk_size = 400` keeps each chunk at roughly **5 % of that limit**, which gives three concrete benefits:

| Property | Value | Rationale |
|---|---|---|
| Chunk size | 400 tokens ≈ 300 words | Large enough to hold a complete article clause or paragraph; small enough to retrieve a precise answer |
| Overlap | 60 tokens ≈ 15 % of chunk | Prevents a sentence that straddles two chunk boundaries from being lost in both |
| Headroom | 8,192 − 400 = 7,792 tokens free | Zero risk of truncation; leaves room for future prompt augmentation |

Larger chunks (e.g. 1,000+ tokens) would embed more context but hurt **retrieval precision** — the model scores the whole chunk, so relevant sentences buried inside long chunks rank lower.  
Smaller chunks (< 200 tokens) risk splitting mid-sentence and losing the grammatical subject of a regulation clause, which is fatal for a legal Q&A use case.  
**400 / 60 is the sweet spot for a dense regulatory document in Spanish.**

In [ ]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Token count ────────────────────────────────────────────────────────────────
enc          = tiktoken.get_encoding("cl100k_base")
total_tokens = len(enc.encode(full_text))

print(f"Total tokens (cl100k_base): {total_tokens:,}")
print(f"Embedding limit            : 8,192 tokens")
print(f"Full doc / limit ratio     : {total_tokens / 8192:.1f}x  \u2192  chunking required\n")

# ── Chunking parameters ────────────────────────────────────────────────────────
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 60

# Use tiktoken so chunk_size means exactly 400 tokens (not 400 characters).
def token_len(text: str) -> int:
    return len(enc.encode(text))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " "],
    length_function=token_len,
)

# ── Metadata attached to every chunk ──────────────────────────────────────────
metadata = {
    "document": PDF_PATH.name,
    "topic"   : "Beca 18 \u2013 Reglamento de Beneficiarios (PRONABEC RDE 033-2026)",
    "language": "es",
}

chunks = splitter.create_documents([full_text], metadatas=[metadata])

# ── Stats ──────────────────────────────────────────────────────────────────────
total_chunks  = len(chunks)
avg_char_len  = sum(len(c.page_content) for c in chunks) / total_chunks
avg_token_len = sum(token_len(c.page_content) for c in chunks) / total_chunks

print(f"{'\u2500'*40}")
print(f"Total chunks      : {total_chunks:>8,}")
print(f"Avg length (chars) : {avg_char_len:>7,.0f}")
print(f"Avg length (tokens): {avg_token_len:>7,.1f}")
print(f"{'\u2500'*40}")

# Preview the first chunk
print(f"\n\u2500\u2500 Chunk 0 preview \u2500\u2500")
print(f"Metadata : {chunks[0].metadata}")
print(f"Content  : {chunks[0].page_content[:300]}...")

---
## Section 6 — Gemini Embedding Functions

### Two task types, two functions

Gemini's `gemini-embedding-001` model is trained with **asymmetric retrieval**: the document and query embedding spaces are optimised differently so that a short question lands close to a long passage that answers it — even though they share few words.

| Function | Task type | Used when |
|---|---|---|
| `embed_documents(texts)` | `RETRIEVAL_DOCUMENT` | Indexing chunks into ChromaDB |
| `embed_query(text)` | `RETRIEVAL_QUERY` | Embedding the user's question at query time |

Using the wrong task type degrades retrieval quality, so both task types are made explicit.

### Exponential backoff with jitter

The Gemini free tier allows ≈ 60 requests per minute. Under load, the API returns HTTP **429 (Resource Exhausted)**. The retry strategy is:

- Base wait: **1 s → 2 s → 4 s → 8 s → 16 s** (doubles each attempt)  
- **Jitter** (`+0–0.5 s` random): prevents multiple concurrent callers from retrying at exactly the same moment (thundering-herd problem)  
- After **5 failed attempts** the exception is re-raised so the notebook fails loudly rather than silently

### Batching

`embed_documents` sends texts in batches of up to **100** per API call and inserts a **1-second pause** between batches, keeping throughput safely below the rate limit even on the free tier.

In [ ]:
import time
import re
import random
from tqdm.notebook import tqdm
from google import genai
from google.genai import types

# ── Gemini client ──────────────────────────────────────────────────────────────
client = genai.Client(api_key=GEMINI_API_KEY)

EMBED_MODEL = "models/gemini-embedding-001"
EMBED_DIM   = 768
BATCH_SIZE  = 100   # texts per API call


# ── Core API call with quota-aware backoff ─────────────────────────────────────
def _call_embed_api(contents: list, task_type: str, max_retries: int = 7) -> list:
    """
    Call the Gemini embedding endpoint.
    - Daily quota (PerDay):    fail immediately with a clear message — retrying
                               won't help until the quota resets at midnight.
    - Per-minute quota:        read the retryDelay hint from the API response
                               and wait that long (+jitter) before retrying.
    """
    fallback_wait = 20.0
    for attempt in range(max_retries):
        try:
            response = client.models.embed_content(
                model=EMBED_MODEL,
                contents=contents,
                config=types.EmbedContentConfig(
                    task_type=task_type,
                    output_dimensionality=EMBED_DIM,
                ),
            )
            return [e.values for e in response.embeddings]
        except Exception as exc:
            exc_str = str(exc)
            is_rate_limit = (
                "429" in exc_str
                or "resource_exhausted" in exc_str.lower()
                or "quota" in exc_str.lower()
            )
            if not is_rate_limit:
                raise

            # Daily quota exhausted — no point retrying until tomorrow
            if "perday" in exc_str.lower() or "per_day" in exc_str.lower():
                raise RuntimeError(
                    "Daily embedding quota exhausted (free tier: 1,000 requests/day).\n"
                    "Please wait until the quota resets at midnight Pacific Time, "
                    "or upgrade your Gemini API plan."
                ) from exc

            # Per-minute rate limit — wait the delay the API suggests
            if attempt < max_retries - 1:
                m = re.search(r"retry in (\d+\.?\d*)s", exc_str, re.IGNORECASE)
                if not m:
                    m = re.search(r"'retryDelay':\s*'(\d+\.?\d*)s'", exc_str)
                delay = float(m.group(1)) + random.uniform(1, 3) if m else fallback_wait + random.uniform(0, 2)
                fallback_wait = min(fallback_wait * 2, 120)
                print(f"  Rate limit — waiting {delay:.1f}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(delay)
            else:
                raise


# ── Public functions ───────────────────────────────────────────────────────────
def embed_documents(texts: list) -> list:
    """Embed texts for indexing using RETRIEVAL_DOCUMENT task type."""
    all_embeddings = []
    batch_starts = list(range(0, len(texts), BATCH_SIZE))
    for i in tqdm(batch_starts, desc="Embedding documents", unit="batch"):
        batch = texts[i : i + BATCH_SIZE]
        all_embeddings.extend(_call_embed_api(batch, task_type="RETRIEVAL_DOCUMENT"))
        if i + BATCH_SIZE < len(texts):
            time.sleep(1.0)
    return all_embeddings


def embed_query(text: str) -> list:
    """Embed a single query string using RETRIEVAL_QUERY task type."""
    return _call_embed_api([text], task_type="RETRIEVAL_QUERY")[0]


# ── Smoke test ─────────────────────────────────────────────────────────────────
print("Running smoke test…")
query_vec = embed_query("¿Qué requisitos necesito para postular a Beca 18?")
print(f"embed_query     → dim: {len(query_vec)},  sample: {[round(v, 6) for v in query_vec[:4]]}")
doc_vecs = embed_documents([chunks[0].page_content, chunks[1].page_content])
print(f"embed_documents → {len(doc_vecs)} vectors, dim: {len(doc_vecs[0])}")
print("\nEmbedding functions ready ✓")


---
## Section 7 — ChromaDB Indexing

### Design decisions

**Cosine distance** is the right metric for semantic search over normalised embedding vectors: it measures the angle between vectors, ignoring magnitude, which aligns with how `gemini-embedding-001` encodes meaning.

**`PersistentClient`** writes the HNSW index to disk at `chroma_db_beca18/`, so the collection survives kernel restarts within the same Colab session. The folder is gitignored (`chroma_db*/`).

**Idempotent indexing** — before embedding any chunk the cell checks `collection.count()`. If documents are already present it skips the entire embedding + upload step. This means:
- Re-running the cell after a kernel restart costs nothing (no API calls, no wait time).
- A fresh session always re-embeds because the Colab filesystem is wiped between sessions.

> **Note:** Colab's filesystem is ephemeral — the `chroma_db_beca18/` folder is lost when the runtime is recycled. The idempotency check protects against accidental double-indexing *within* a session, not across sessions.

In [ ]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"   # must be set before importing chromadb

import chromadb

# ── Persistent client & collection ────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(
    path="chroma_db_beca18",
    settings=chromadb.Settings(anonymized_telemetry=False),
)

collection = chroma_client.get_or_create_collection(
    name="beca18_reglamento",
    metadata={"hnsw:space": "cosine"},
)

# ── Idempotency check ──────────────────────────────────────────────────────────
existing_count = collection.count()

if existing_count > 0:
    print(f"Collection already contains {existing_count:,} documents — skipping embedding.")
else:
    texts     = [c.page_content for c in chunks]
    metadatas = [c.metadata     for c in chunks]
    ids       = [f"chunk_{i}"  for i in range(len(chunks))]

    print(f"Embedding and indexing {len(chunks):,} chunks — this may take a few minutes…")
    embeddings = embed_documents(texts)

    UPSERT_BATCH = 500
    for start in range(0, len(chunks), UPSERT_BATCH):
        end = start + UPSERT_BATCH
        collection.add(
            documents=texts[start:end],
            embeddings=embeddings[start:end],
            metadatas=metadatas[start:end],
            ids=ids[start:end],
        )
    print("Indexing complete.")

# ── Confirm ───────────────────────────────────────────────────────────────────
total = collection.count()
print(f"{'─'*40}")
print(f"Documents in collection : {total:>6,}")
print(f"Collection name         : {collection.name}")
print(f"Distance metric         : cosine")
print(f"{'─'*40}")


---
## Section 8 — Semantic Search

`semantic_search` embeds the question with `RETRIEVAL_QUERY` task type and queries
the ChromaDB collection for the *k* closest chunks by cosine distance.

> **Cosine distance interpretation:** 0 = identical direction, 1 = orthogonal, 2 = opposite.  
> Scores below ~0.3 are strong matches for this embedding model.

In [ ]:
def semantic_search(question: str, k: int = 5) -> list:
    """
    Return the top-k chunks most semantically similar to `question`.

    Each result dict contains:
      - text     : the raw chunk text
      - metadata : document / topic / language fields
      - distance : cosine distance (lower = more similar)
    """
    query_vec = embed_query(question)

    results = collection.query(
        query_embeddings=[query_vec],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )

    return [
        {
            "text"    : doc,
            "metadata": meta,
            "distance": round(dist, 4),
        }
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


# ── Test ──────────────────────────────────────────────────────────────────────
SAMPLE_QUESTION = "¿Cuáles son los requisitos económicos para postular a Beca 18?"

hits = semantic_search(SAMPLE_QUESTION, k=3)

print(f'Question: "{SAMPLE_QUESTION}"')
print(f"{'─'*70}\n")

for i, hit in enumerate(hits, start=1):
    print(f"[{i}] Distance : {hit['distance']}")
    print(f"    Document : {hit['metadata']['document']}")
    print(f"    Text     : {hit['text'][:300].strip()}...")
    print()


---
## Section 9 — RAG Answer Generation

`answer_with_context` follows the standard RAG pattern:
**retrieve → augment → generate**.

| Step | What happens |
|---|---|
| Retrieve | `semantic_search` embeds the question and fetches the top-*k* chunks from ChromaDB |
| Augment | The chunks are assembled into a numbered context block and injected into the prompt |
| Generate | `gemini-2.5-flash` answers using **only** the provided context |

### System prompt guarantees
- **Grounding:** the model must not use any knowledge outside the retrieved passages.
- **Citation:** when a `[PAGE N]` marker is visible in the context the model must reference it.
- **Refusal:** if the context is insufficient the model must reply with the exact phrase
  *"The document does not contain information about this topic."* — preventing hallucination.
- **Language:** the answer is always in the same language as the question.

In [ ]:
from google.genai import types

GENERATION_MODEL = "gemini-2.5-flash"

SYSTEM_PROMPT = """\
You are a precise assistant specialized in the Beca 18 scholarship regulations
(PRONABEC RDE 033-2026). Answer questions from students and advisors.

Rules you must follow without exception:
1. Base your answer EXCLUSIVELY on the context passages provided. Do not use any prior or external knowledge.
2. Whenever relevant information appears on a specific page, cite it using the [PAGE N] marker
   visible in the passage (e.g., "According to page 7...").
3. If the context does not contain sufficient information to answer the question, respond with
   exactly: "The document does not contain information about this topic."
4. Never speculate, infer, or add anything not explicitly stated in the provided context.
5. Respond in the same language the question is written in.
"""


def answer_with_context(question: str, k: int = 5) -> str:
    """Retrieve the top-k chunks for `question` and generate a grounded answer."""
    hits = semantic_search(question, k=k)

    context = "\n\n".join(
        f"--- Passage {i} (cosine distance: {h['distance']}) ---\n{h['text']}"
        for i, h in enumerate(hits, 1)
    )

    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=f"Context:\n{context}\n\nQuestion: {question}",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.0,
        ),
    )
    return response.text


print("answer_with_context ready.")


In [ ]:
test_cases = [
    ("Eligibility requirements",         "¿Cuáles son los requisitos para postular a Beca 18?"),
    ("Scholarship modalities",            "¿Cuáles son las modalidades de Beca 18 y en qué se diferencian?"),
    ("Monthly stipend",                   "¿Cuál es el monto de la subvención económica mensual que recibe un becario?"),
    ("Student obligations",               "¿Cuáles son las obligaciones del becario durante la vigencia de la beca?"),
    ("Conditions for losing scholarship", "¿Bajo qué condiciones se pierde o cancela la Beca 18?"),
    ("Off-topic (refusal test)",          "¿Cuáles son los mejores restaurantes en Lima para celebrar una graduación?"),
]

for label, question in test_cases:
    print(f"{'═'*70}")
    print(f"  Topic    : {label}")
    print(f"  Question : {question}")
    print(f"{'─'*70}")
    answer = answer_with_context(question, k=5)
    print(answer)
    print()


---
## Section 10 — Chat Interface

Interactive ipywidgets UI with:
- **Text box** — type your question about Beca 18
- **Ask / Clear** buttons
- **k slider** — controls how many chunks are retrieved from ChromaDB
- **Answer panel** — displays the grounded answer
- **Source accordion** — expandable list of retrieved fragments with page number and cosine distance

In [ ]:
import re
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Input widgets ─────────────────────────────────────────────────────────────
question_box = widgets.Text(
    placeholder="Escribe tu pregunta sobre Beca 18…",
    layout=widgets.Layout(width="68%", height="36px"),
)

ask_btn = widgets.Button(
    description="Ask",
    button_style="primary",
    icon="search",
    layout=widgets.Layout(width="90px"),
)

clear_btn = widgets.Button(
    description="Clear",
    button_style="warning",
    icon="times",
    layout=widgets.Layout(width="90px"),
)

k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description="k chunks:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="380px"),
)

# ── Output areas ──────────────────────────────────────────────────────────────
answer_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="12px",
        min_height="60px",
        margin="8px 0",
        border_radius="6px",
    )
)

sources_output = widgets.Output()
sources_accordion = widgets.Accordion(children=[sources_output])
sources_accordion.set_title(0, "Source fragments")
sources_accordion.selected_index = None  # collapsed by default

# ── Layout ────────────────────────────────────────────────────────────────────
input_row = widgets.HBox(
    [question_box, ask_btn, clear_btn],
    layout=widgets.Layout(gap="8px", align_items="center"),
)

ui = widgets.VBox(
    [input_row, k_slider, answer_output, sources_accordion],
    layout=widgets.Layout(width="100%", padding="8px"),
)


# ── Ask callback ──────────────────────────────────────────────────────────────
def on_ask(b):
    question = question_box.value.strip()
    if not question:
        return

    ask_btn.disabled = True
    ask_btn.description = "Thinking…"

    with answer_output:
        clear_output()
        display(widgets.HTML("<i>Searching context and generating answer…</i>"))

    k = k_slider.value

    try:
        hits   = semantic_search(question, k=k)
        answer = answer_with_context(question, k=k)
    except Exception as exc:
        with answer_output:
            clear_output()
            display(widgets.HTML(f"<b style='color:red'>Error:</b> {exc}"))
        ask_btn.disabled = False
        ask_btn.description = "Ask"
        return

    # ── Render answer ──────────────────────────────────────────────────────────
    with answer_output:
        clear_output()
        display(widgets.HTML(
            "<b style='font-size:1.05em'>Answer</b>"
            "<hr style='margin:4px 0 10px'>"
            + answer.replace("\n", "<br>")
        ))

    # ── Render source fragments ────────────────────────────────────────────────
    with sources_output:
        clear_output()
        for i, hit in enumerate(hits, 1):
            page_m  = re.search(r"\[PAGE (\d+)\]", hit["text"])
            page_lbl = f"Page {page_m.group(1)}" if page_m else "Page unknown"
            preview  = hit["text"][:450].replace("\n", " ").strip()
            display(widgets.HTML(
                f"<div style='margin-bottom:10px; padding:10px 12px; "
                f"border-left:3px solid #4A90D9; background:#f9f9f9; border-radius:0 4px 4px 0'>"
                f"<b>[{i}]</b> &nbsp; 📄 {page_lbl} &nbsp;"
                f"<span style='color:#888; font-size:0.9em'>| distance: {hit['distance']}</span><br>"
                f"<span style='font-size:0.91em; color:#333'>{preview}…</span>"
                f"</div>"
            ))

    sources_accordion.set_title(0, f"Source fragments  ({k} chunks retrieved)")
    ask_btn.disabled = False
    ask_btn.description = "Ask"


# ── Clear callback ────────────────────────────────────────────────────────────
def on_clear(b):
    question_box.value = ""
    with answer_output:
        clear_output()
    with sources_output:
        clear_output()
    sources_accordion.set_title(0, "Source fragments")
    sources_accordion.selected_index = None


ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)

display(ui)